# ARC-AGI-3 A/B: Duck harness vs Duck + arc3 tricks

Harness: TAAF/Tufa Labs Duck (CC0 bundle); model Qwen3.8-27B-FP8 (Apache 2.0); wrapper from the public notebook foysalemonshanto/lb-9-arc3-duck-v12-with-qwen-3-8-27b.

**Save run (not scored):** 8 public games x 2 passes. Pass 0 = plain Duck; pass 1 = Duck + two arc3-autoresearch tricks: (1) a no-op guard that ignores timer/step-counter border strips, (2) `nav_moves()` / `nav_path()` sandbox helpers that learn the player sprite and plan arrow paths. Output: `arc3_ab_report.md` / `.json` with levels, actions, score, time, tokens and trick counters per game and arm.

**Competition rerun (scored):** every hidden game plays with the tricks on (ARC3_MODE=on).


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


def _patch_qwen38_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    Qwen3.8 Kaggle Model. This avoids copying/forking the large bundled setup
    script and keeps the wheelhouse/GPU/vLLM behavior from the source bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
    }

    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for Qwen3.8. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: Qwen3.8 setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_qwen38_setup_commands(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use Qwen3.8.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {QWEN_SERVED_MODEL_NAME!r}"
    )

print("\n✅ TAAF/vLLM setup completed for Qwen3.8")
print("Model path:", QWEN_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
# arc3 tricks: write the add-on, choose the arm policy, install before the benchmark is unpickled.
import importlib

ARC3_TRICKS_SOURCE = "\"\"\"arc3_tricks -- two arc3-autoresearch tricks bolted onto the Duck harness, A/B-able.\n\n1. HUD-aware no-op guard. Duck's guard blocks an action only when it already did\n   nothing on the *identical* board. A timer or step counter changes the board on\n   every action, so on those games the guard never fires. Here, border strips\n   (rows/cols within EDGE of the frame edge) that change on at least FRAC of the\n   observed actions are ignored when deciding \"did anything happen\" and when keying\n   the guard's board signature.\n2. Navigation helpers inside the model's Python sandbox: nav_moves() learns the\n   player sprite, each arrow's displacement and the colours it has walked onto\n   from past transitions; nav_path(cell) returns the shortest arrow sequence that\n   puts the player on a cell. They cost no game actions.\n\nArms: ARC3_MODE=ab -> pass 0 plays plain Duck, pass 1 plays with the tricks;\nARC3_MODE=on -> every run gets the tricks; ARC3_MODE=off -> plain Duck everywhere.\nPer-run counters are written next to the harness artifacts as <run>_arc3.json.\n\nApply with install() after the TAAF sources are importable and before\nbenchmark_initial.pkl is unpickled.\n\"\"\"\nfrom __future__ import annotations\n\nimport inspect\nimport json\nimport os\nimport textwrap\nfrom collections import Counter\nfrom pathlib import Path\n\nEDGE = 3        # a HUD strip must lie within this many cells of the frame edge\nFRAC = 0.6      # ...and change on at least this fraction of actions\nMIN_OBS = 4     # ...once at least this many actions have been seen\nMAX_LINES = 8   # never mask more strips than this (a sprite hugging a wall is not a HUD)\n\nADDENDUM = (\n    \"\\n\\nExtra harness helpers inside `python` (they cost no game actions):\\n\"\n    \"- `nav_moves()` learns from past UP/DOWN/LEFT/RIGHT transitions which object is the player, \"\n    \"how far each arrow moves it, and which colours it has walked onto. It returns a dict with \"\n    \"`player_color`, `size`, `position` (top-left `[row, col]` now), `moves` (`{'UP': [dr, dc], ...}`), \"\n    \"`walkable_colors` and `blocked` (arrows that did nothing), or `None` if no arrow has visibly moved an object yet.\\n\"\n    \"- `nav_path((row, col), walkable=None, max_nodes=40000)` returns `{'actions': [...], 'end': [row, col], 'nodes': n}`: \"\n    \"the shortest arrow sequence after which the player covers cell `(row, col)`, stepping only on colours it \"\n    \"has already walked onto (pass `walkable` as a list of colour characters to override). `actions` is `None` \"\n    \"when no path exists under that model. Check the result, then execute it with `action(...)`; if the board \"\n    \"reacts differently from the prediction, re-plan.\\n\"\n    \"- The no-op guard ignores border strips that change on almost every action (timers, step counters), so an \"\n    \"action whose only effect was a ticking timer is blocked like any other known no-op in that state.\\n\"\n)\n\n_HELPERS = r'''\nfrom collections import Counter\n\n_ARC3_ARROWS = {\"UP\": \"UP\", \"DOWN\": \"DOWN\", \"LEFT\": \"LEFT\", \"RIGHT\": \"RIGHT\",\n                \"ACTION1\": \"UP\", \"ACTION2\": \"DOWN\", \"ACTION3\": \"LEFT\", \"ACTION4\": \"RIGHT\"}\n\n\ndef _arc3_arrow(name):\n    return _ARC3_ARROWS.get(str(name or \"\").strip().split(\" \")[0].upper())\n\n\ndef _arc3_grid(frame):\n    return getattr(frame, \"_grid\", None) or []\n\n\ndef _arc3_components(grid, color, seeds):\n    \"\"\"4-connected components of `color` that contain at least one seed cell.\"\"\"\n    rows = len(grid)\n    cols = len(grid[0]) if rows else 0\n    seen = set()\n    out = []\n    for sr, sc in seeds:\n        if (sr, sc) in seen or not (0 <= sr < rows and 0 <= sc < cols) or grid[sr][sc] != color:\n            continue\n        stack = [(sr, sc)]\n        seen.add((sr, sc))\n        cells = []\n        while stack:\n            r, c = stack.pop()\n            cells.append((r, c))\n            if len(cells) > 400:\n                break\n            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):\n                if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in seen and grid[nr][nc] == color:\n                    seen.add((nr, nc))\n                    stack.append((nr, nc))\n        if len(cells) <= 400:\n            r0 = min(r for r, _ in cells)\n            c0 = min(c for _, c in cells)\n            out.append(((r0, c0), frozenset((r - r0, c - c0) for r, c in cells)))\n    return out\n\n\ndef _arc3_model(globals_):\n    votes = {}\n    moved_onto = {}\n    blocked = Counter()\n    last_pos = {}\n    for t in globals_.get(\"transitions\") or []:\n        arrow = _arc3_arrow(getattr(t, \"action\", \"\"))\n        if arrow is None or t.before_frame is None or t.after_frame is None:\n            continue\n        before = _arc3_grid(t.before_frame)\n        after = _arc3_grid(t.after_frame)\n        if not before or not after or len(before) != len(after):\n            continue\n        changed = [(r, c) for r in range(len(before)) for c in range(len(before[r]))\n                   if before[r][c] != after[r][c]]\n        if not changed:\n            blocked[arrow] += 1\n            continue\n        if len(changed) > 600:\n            continue\n        colours = {before[r][c] for r, c in changed} | {after[r][c] for r, c in changed}\n        for colour in colours:\n            seeds = [(r, c) for r, c in changed if before[r][c] == colour or after[r][c] == colour]\n            b_comps = _arc3_components(before, colour, seeds)\n            a_comps = _arc3_components(after, colour, seeds)\n            for b_origin, b_shape in b_comps:\n                for a_origin, a_shape in a_comps:\n                    if a_shape != b_shape:\n                        continue\n                    dr, dc = a_origin[0] - b_origin[0], a_origin[1] - b_origin[1]\n                    if (dr, dc) == (0, 0) or abs(dr) > 16 or abs(dc) > 16:\n                        continue\n                    key = (colour, b_shape)\n                    votes.setdefault(key, {}).setdefault(arrow, Counter())[(dr, dc)] += 1\n                    last_pos[key] = a_origin\n                    for rr, cc in b_shape:\n                        r, c = a_origin[0] + rr, a_origin[1] + cc\n                        if (rr + dr, cc + dc) not in b_shape and 0 <= r < len(before) and 0 <= c < len(before[0]):\n                            moved_onto.setdefault(key, Counter())[before[r][c]] += 1\n    if not votes:\n        return None\n    key = max(votes, key=lambda k: (sum(sum(v.values()) for v in votes[k].values()), len(votes[k])))\n    colour, shape = key\n    moves = {arrow: list(counter.most_common(1)[0][0]) for arrow, counter in votes[key].items()}\n    grid = _arc3_grid(globals_.get(\"current_frame\"))\n    position = None\n    if grid:\n        best = None\n        ar, ac = min(shape)\n        for r in range(len(grid)):\n            for c in range(len(grid[r])):\n                if grid[r][c] != colour:\n                    continue\n                o_r, o_c = r - ar, c - ac\n                if all(0 <= o_r + rr < len(grid) and 0 <= o_c + cc < len(grid[0])\n                       and grid[o_r + rr][o_c + cc] == colour for rr, cc in shape):\n                    lr, lc = last_pos.get(key, (o_r, o_c))\n                    d = abs(o_r - lr) + abs(o_c - lc)\n                    if best is None or d < best[0]:\n                        best = (d, (o_r, o_c))\n        position = list(best[1]) if best else None\n    walk = sorted(moved_onto.get(key, Counter()))\n    return {\n        \"player\": colour,\n        \"shape\": shape,\n        \"position\": position,\n        \"moves\": moves,\n        \"walkable\": walk,\n        \"blocked\": sorted(a for a in blocked if a not in moves),\n    }\n\n\ndef _arc3_char(value):\n    try:\n        return COLOR_CHARS[int(value)]\n    except Exception:\n        return str(value)\n\n\ndef nav_moves(globals_):\n    m = _arc3_model(globals_)\n    if m is None:\n        return None\n    h = max(r for r, _ in m[\"shape\"]) + 1\n    w = max(c for _, c in m[\"shape\"]) + 1\n    return {\n        \"player_color\": _arc3_char(m[\"player\"]),\n        \"size\": [h, w],\n        \"position\": m[\"position\"],\n        \"moves\": m[\"moves\"],\n        \"walkable_colors\": [_arc3_char(v) for v in m[\"walkable\"]],\n        \"blocked\": m[\"blocked\"],\n    }\n\n\ndef nav_path(globals_, target, walkable=None, max_nodes=40000):\n    m = _arc3_model(globals_)\n    if m is None or m[\"position\"] is None:\n        return {\"actions\": None, \"reason\": \"no player model yet: make an arrow move that visibly moves an object\"}\n    grid = _arc3_grid(globals_.get(\"current_frame\"))\n    rows, cols = len(grid), len(grid[0]) if grid else 0\n    tr, tc = int(target[0]), int(target[1])\n    if walkable is None:\n        ok = set(m[\"walkable\"])\n    else:\n        chars = {str(v) for v in walkable}\n        ok = {i for i, ch in enumerate(COLOR_CHARS) if ch in chars} | {int(v) for v in walkable if str(v).isdigit()}\n    ok.add(m[\"player\"])\n    start = tuple(m[\"position\"])\n    here = {(start[0] + rr, start[1] + cc) for rr, cc in m[\"shape\"]}\n\n    def fits(pos):\n        for rr, cc in m[\"shape\"]:\n            r, c = pos[0] + rr, pos[1] + cc\n            if not (0 <= r < rows and 0 <= c < cols):\n                return False\n            if (r, c) not in here and grid[r][c] not in ok:\n                return False\n        return True\n\n    def covers(pos):\n        return (tr - pos[0], tc - pos[1]) in m[\"shape\"]\n\n    parent = {start: None}\n    frontier = [start]\n    nodes = 0\n    while frontier and nodes < max_nodes:\n        nxt = []\n        for pos in frontier:\n            nodes += 1\n            if covers(pos):\n                path = []\n                cur = pos\n                while parent[cur] is not None:\n                    prev, arrow = parent[cur]\n                    path.append(arrow)\n                    cur = prev\n                return {\"actions\": path[::-1], \"end\": list(pos), \"nodes\": nodes}\n            for arrow, (dr, dc) in m[\"moves\"].items():\n                npos = (pos[0] + dr, pos[1] + dc)\n                if npos not in parent and fits(npos):\n                    parent[npos] = (pos, arrow)\n                    nxt.append(npos)\n        frontier = nxt\n    return {\"actions\": None, \"reason\": \"no path under the learned moves and walkable colours\", \"nodes\": nodes}\n\n'''\n\n_REGISTER = (\n    '    runtime_globals[\"action\"] = action\\n'\n    '    if (initial.get(\"state\") or {}).get(\"arc3_helpers\"):\\n'\n    '        runtime_globals[\"nav_moves\"] = lambda: nav_moves(runtime_globals)\\n'\n    '        runtime_globals[\"nav_path\"] = lambda target, walkable=None, max_nodes=40000: '\n    'nav_path(runtime_globals, target, walkable, max_nodes)\\n'\n)\n\n_RUN_PATCHES = [\n    (\"noop_guard_board_sig = board_signature(current_frame.grid) if current_frame is not None else board_signature(())\",\n     \"noop_guard_board_sig = self._arc3_sig(current_frame.grid if current_frame is not None else ())\\n\"\n     \"{I}noop_guard_grid = current_frame.grid if current_frame is not None else ()\", 1),\n    (\"nonlocal terminal_action_result, noop_guard_board_sig, noop_guard_level\",\n     \"nonlocal terminal_action_result, noop_guard_board_sig, noop_guard_level, noop_guard_grid\", 1),\n    ('board_changed=bool(compact_payload.get(\"board_changed\")),',\n     'board_changed=self._arc3_changed(compact_payload, noop_guard_grid, state_path),', 1),\n    ('board_changed=bool(sub_compact.get(\"board_changed\")),',\n     'board_changed=self._arc3_changed(sub_compact, noop_guard_grid, state_path),', 1),\n    (\"noop_guard_board_sig = board_signature(refreshed_frame.grid)\",\n     \"noop_guard_board_sig = self._arc3_sig(refreshed_frame.grid)\\n\"\n     \"{I}noop_guard_grid = refreshed_frame.grid\", 2),\n    (\"blocked_actions.append(action_sig)\",\n     \"blocked_actions.append(action_sig)\\n{I}self._arc3_count('guard_blocks')\", 1),\n    (\"last_result = self._last_action_result or {}\",\n     \"self._arc3_count('guard_blocks')\\n{I}last_result = self._last_action_result or {}\", 1),\n    (\"initial_state=_serialized_runtime_state(),\",\n     \"initial_state=self._arc3_state(_serialized_runtime_state()),\", 1),\n]\n\n\ndef arm_for_pass(pass_index: int) -> bool:\n    mode = os.environ.get(\"ARC3_MODE\", \"ab\").strip().lower()\n    if mode == \"on\":\n        return True\n    if mode == \"off\":\n        return False\n    return int(pass_index) % 2 == 1\n\n\ndef _patch(text: str, old: str, new: str, count: int) -> str:\n    \"\"\"Replace `old` exactly `count` times; `{I}` in `new` becomes the matched line's indentation.\"\"\"\n    found = text.count(old)\n    if found != count:\n        raise RuntimeError(f\"arc3_tricks: expected {count} x {old[:60]!r}, found {found}\")\n    out, pos = [], 0\n    while True:\n        at = text.find(old, pos)\n        if at < 0:\n            break\n        line_start = text.rfind(\"\\n\", 0, at) + 1\n        indent = text[line_start:at]\n        indent = indent if not indent.strip() else \"\"\n        out.append(text[pos:at])\n        out.append(new.replace(\"{I}\", indent))\n        pos = at + len(old)\n    out.append(text[pos:])\n    return \"\".join(out)\n\n\ndef install() -> dict:\n    from inference.agent import python_tool_sandbox as sandbox\n    from inference.agent import tool_agent as ta\n    from inference.framework import solver as solver_mod\n\n    if getattr(ta.ToolAgent, \"_arc3_installed\", False):\n        return {\"installed\": \"already\"}\n\n    # 1) sandbox helpers\n    boot = sandbox._SANDBOX_BOOTSTRAP\n    boot = _patch(boot, \"\\ndef main():\\n\", \"\\n\" + _HELPERS + \"\\ndef main():\\n\", 1)\n    boot = _patch(boot, '    runtime_globals[\"action\"] = action\\n', _REGISTER, 1)\n    compile(boot, \"<arc3_bootstrap>\", \"exec\")\n    sandbox._SANDBOX_BOOTSTRAP = boot\n\n    # 2) guard + helper flag inside ToolAgent._run_python_tool\n    src = textwrap.dedent(inspect.getsource(ta.ToolAgent._run_python_tool))\n    for old, new, count in _RUN_PATCHES:\n        src = _patch(src, old, new, count)\n    namespace: dict = {}\n    exec(compile(src, \"<arc3_run_python_tool>\", \"exec\"), ta.__dict__, namespace)\n    ta.ToolAgent._run_python_tool_plain = ta.ToolAgent._run_python_tool\n    patched_run = namespace[\"_run_python_tool\"]\n\n    def _arc3_run_python_tool(self, state_path, arguments):\n        code = str((arguments or {}).get(\"code\", \"\"))\n        if getattr(self, \"_arc3_on\", False):\n            if \"nav_moves(\" in code:\n                self._arc3_count(\"nav_moves_calls\")\n            if \"nav_path(\" in code:\n                self._arc3_count(\"nav_path_calls\")\n        self._arc3_count(\"python_calls\")\n        try:\n            return patched_run(self, state_path, arguments)\n        finally:\n            self._arc3_flush()\n\n    def _arc3_count(self, key, amount=1):\n        stats = self.__dict__.setdefault(\"_arc3_stats\", Counter())\n        stats[key] += amount\n\n    def _arc3_masked_lines(self, shape):\n        rows, cols = shape\n        n = self.__dict__.get(\"_arc3_n\", 0)\n        if n < MIN_OBS or rows == 0:\n            return set(), set()\n        rc = self.__dict__.get(\"_arc3_rows\", Counter())\n        cc = self.__dict__.get(\"_arc3_cols\", Counter())\n        mr = {r for r, k in rc.items() if k >= FRAC * n and (r < EDGE or r >= rows - EDGE)}\n        mc = {c for c, k in cc.items() if k >= FRAC * n and (c < EDGE or c >= cols - EDGE)}\n        if len(mr) + len(mc) > MAX_LINES:\n            return set(), set()\n        return mr, mc\n\n    def _arc3_sig(self, grid):\n        if not getattr(self, \"_arc3_on\", False) or not grid:\n            return ta.board_signature(grid)\n        mr, mc = self._arc3_masked_lines((len(grid), len(grid[0])))\n        if not mr and not mc:\n            return ta.board_signature(grid)\n        masked = [[-1 if (r in mr or c in mc) else v for c, v in enumerate(row)] for r, row in enumerate(grid)]\n        return ta.board_signature(masked)\n\n    def _arc3_changed(self, payload, before, state_path):\n        env_changed = bool(payload.get(\"board_changed\"))\n        if not getattr(self, \"_arc3_on\", False):\n            return env_changed\n        after_frame, _ = ta.load_runtime_state(state_path)\n        after = after_frame.grid if after_frame is not None else ()\n        if not before or not after or len(before) != len(after):\n            return env_changed\n        diff = [(r, c) for r in range(len(after)) for c in range(len(after[r]))\n                if r < len(before) and c < len(before[r]) and before[r][c] != after[r][c]]\n        self.__dict__[\"_arc3_n\"] = self.__dict__.get(\"_arc3_n\", 0) + 1\n        rc = self.__dict__.setdefault(\"_arc3_rows\", Counter())\n        cc = self.__dict__.setdefault(\"_arc3_cols\", Counter())\n        for r in {r for r, _ in diff}:\n            rc[r] += 1\n        for c in {c for _, c in diff}:\n            cc[c] += 1\n        if env_changed and diff:\n            mr, mc = self._arc3_masked_lines((len(after), len(after[0])))\n            if (mr or mc) and all(r in mr or c in mc for r, c in diff):\n                self._arc3_count(\"hud_only_changes\")\n                return False\n        return env_changed\n\n    def _arc3_state(self, state):\n        if getattr(self, \"_arc3_on\", False) and isinstance(state, dict):\n            state = dict(state)\n            state[\"arc3_helpers\"] = True\n        return state\n\n    def _arc3_flush(self):\n        path = getattr(self, \"_arc3_stats_path\", None)\n        if path is None:\n            return\n        shape = (64, 64)\n        mr, mc = self._arc3_masked_lines(shape)\n        data = {\n            \"arm\": \"tricks\" if getattr(self, \"_arc3_on\", False) else \"plain\",\n            \"counters\": dict(self.__dict__.get(\"_arc3_stats\", Counter())),\n            \"actions_observed\": self.__dict__.get(\"_arc3_n\", 0),\n            \"hud_rows\": sorted(mr),\n            \"hud_cols\": sorted(mc),\n        }\n        try:\n            Path(path).write_text(json.dumps(data, indent=1), encoding=\"utf-8\")\n        except OSError:\n            pass\n\n    for name, fn in [(\"_run_python_tool\", _arc3_run_python_tool), (\"_arc3_count\", _arc3_count),\n                     (\"_arc3_masked_lines\", _arc3_masked_lines), (\"_arc3_sig\", _arc3_sig),\n                     (\"_arc3_changed\", _arc3_changed), (\"_arc3_state\", _arc3_state),\n                     (\"_arc3_flush\", _arc3_flush)]:\n        setattr(ta.ToolAgent, name, fn)\n    ta.ToolAgent._arc3_installed = True\n\n    # 3) choose the arm per run when the game session starts\n    session_cls = solver_mod._HarnessGameSession\n    plain_play = session_cls.play\n\n    def play(self, *args, **kwargs):\n        agent = self.analyzer\n        on = arm_for_pass(self.pass_index)\n        agent._arc3_on = on\n        agent._arc3_stats_path = Path(str(self.state_path).replace(ta.RUNTIME_STATE_FILENAME, \"arc3.json\"))\n        if on and ADDENDUM not in agent._system_prompt:\n            agent._system_prompt = agent._system_prompt + ADDENDUM\n        agent._arc3_flush()\n        return plain_play(self, *args, **kwargs)\n\n    session_cls.play = play\n    return {\"installed\": True, \"mode\": os.environ.get(\"ARC3_MODE\", \"ab\")}\n"
Path('/kaggle/working/arc3_tricks.py').write_text(ARC3_TRICKS_SOURCE, encoding='utf-8')
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')
# Save run: pass 0 plain Duck, pass 1 Duck + tricks. Competition rerun: tricks on every game.
os.environ['ARC3_MODE'] = 'on' if _env_bool('KAGGLE_IS_COMPETITION_RERUN', False) else 'ab'
import arc3_tricks
arc3_tricks = importlib.reload(arc3_tricks)
print('arc3_tricks:', arc3_tricks.install())


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Inline customization hook — Q38 P1-style public evaluation.
#
# Q38 P1 runs the full 25 public ARC-AGI-3 games once each (25 games × 1 pass).
# This override applies only to the public/offline notebook run. Competition reruns
# still replace bm.games from Kaggle's live gateway in the final run cell.

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))
print("Qwen3.8 model path:", os.environ.get("TAAF_QWEN_MODEL_PATH"))

Q38_P1_PUBLIC_GAME_IDS = [
    "ls20-9607627b",
    "g50t-5849a774",
    "tu93-0768757b",
    "dc22-fdcac232",
    "cd82-fb555c5d",
    "vc33-5430563c",
    "tn36-ef4dde99",
    "sp80-589a99af",
]

if not true_submission:
    if len(Q38_P1_PUBLIC_GAME_IDS) < 1 or len(set(Q38_P1_PUBLIC_GAME_IDS)) != len(Q38_P1_PUBLIC_GAME_IDS):
        raise RuntimeError("Public smoke game list must contain unique games.")
    if not bm.games:
        raise RuntimeError("benchmark_initial.pkl contains no template public game.")

    import taaf.game_api

    template_game = bm.games[0]
    arcade_spec = getattr(template_game, "arcade_spec", None)
    if arcade_spec is None:
        arcade_spec = getattr(template_game, "_arcade_spec", None)
    if arcade_spec is None:
        raise RuntimeError(
            "Could not recover the public ArcadeSpec from benchmark_initial.pkl; "
            "cannot construct the 25-game Q38 P1 evaluation set."
        )

    # A/B: the list twice; the solver numbers repeats of a game as pass 0, 1.
    bm.games = [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=arcade_spec)
        for _ in range(2)
        for game_id in Q38_P1_PUBLIC_GAME_IDS
    ]
    bm.n_passes = 1
    bm.game_weights = None

    # These already match Q38 P1 in the source notebook; set them explicitly so the
    # intended evaluation configuration is visible and stable.
    if hasattr(bm.solver, "concurrency"):
        bm.solver.concurrency = 16
    if hasattr(bm.solver, "max_runtime_s_per_game"):
        bm.solver.max_runtime_s_per_game = 2700.0

    bm.label = f"{bm.label}-arc3-ab-8g"
    print(f"Public evaluation override: {len(bm.games)} games × {bm.n_passes} pass = {len(bm.games) * bm.n_passes} runs")
    print("Public evaluation concurrency:", getattr(bm.solver, "concurrency", None))
    print("Public per-game runtime cap (s):", getattr(bm.solver, "max_runtime_s_per_game", None))


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
# A/B report: plain Duck (pass 0) vs Duck + arc3 tricks (pass 1)
if not true_submission:
    import collections

    bench = json.loads((WORKING_DIR / 'benchmark.json').read_text(encoding='utf-8'))
    seen = collections.Counter()
    rows = []
    for run in bench.get('game_runs', []):
        gid = run.get('game_id', '')
        p = seen[gid]
        seen[gid] += 1
        stats_files = sorted((WORKING_DIR / 'artifacts').glob(f'{gid}*_p{p}_arc3.json'))
        stats = json.loads(stats_files[0].read_text()) if stats_files else {}
        tokens = re.search(r'tokens=(\d+)', str(run.get('solver_note', '')))
        rows.append({
            'game': gid.split('-')[0],
            'pass': p,
            'arm': stats.get('arm', 'tricks' if p % 2 else 'plain'),
            'levels': run.get('levels_completed', 0) or 0,
            'n_levels': run.get('number_of_levels'),
            'actions': len(run.get('history') or []),
            'actions_per_level': run.get('actions_per_level'),
            'human_per_level': run.get('base_actions_per_level'),
            'score': round(float(run.get('final_score') or 0), 3),
            'state': run.get('state'),
            'seconds': round(float(run.get('final_wallclock_seconds') or 0)),
            'tokens': int(tokens.group(1)) if tokens else None,
            'counters': stats.get('counters', {}),
            'hud_rows': stats.get('hud_rows'),
            'hud_cols': stats.get('hud_cols'),
        })

    def arm_summary(arm):
        rs = [r for r in rows if r['arm'] == arm]
        return {
            'runs': len(rs),
            'mean_score': round(sum(r['score'] for r in rs) / max(1, len(rs)), 3),
            'levels': sum(r['levels'] for r in rs),
            'games_with_a_level': sum(1 for r in rs if r['levels']),
            'actions': sum(r['actions'] for r in rs),
            'tokens': sum(r['tokens'] or 0 for r in rs),
            'seconds': sum(r['seconds'] for r in rs),
        }

    summary = {arm: arm_summary(arm) for arm in ('plain', 'tricks')}
    report = {
        'label': bench.get('label'),
        'summary': summary,
        'runs': rows,
        'note': 'One pass per arm: per-game differences are anecdotes; look for a consistent direction.',
    }
    (WORKING_DIR / 'arc3_ab_report.json').write_text(json.dumps(report, indent=1), encoding='utf-8')
    lines = ['# arc3 A/B report', '',
             '| arm | runs | mean score | levels | games with a level | actions | tokens | seconds |',
             '|---|---|---|---|---|---|---|---|']
    for arm, s in summary.items():
        lines.append(f"| {arm} | {s['runs']} | {s['mean_score']} | {s['levels']} | {s['games_with_a_level']} "
                     f"| {s['actions']} | {s['tokens']} | {s['seconds']} |")
    lines += ['',
              '| game | arm | levels | actions per level | human per level | score | state | s | tokens | trick counters | HUD rows/cols |',
              '|---|---|---|---|---|---|---|---|---|---|---|']
    for r in sorted(rows, key=lambda r: (r['game'], r['pass'])):
        lines.append(f"| {r['game']} | {r['arm']} | {r['levels']}/{r['n_levels']} | {r['actions_per_level']} "
                     f"| {r['human_per_level']} | {r['score']} | {r['state']} | {r['seconds']} | {r['tokens']} "
                     f"| {r['counters']} | {r['hud_rows']}/{r['hud_cols']} |")
    (WORKING_DIR / 'arc3_ab_report.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    print('\n'.join(lines))


In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")